In [5]:
# probe
from dorna2 import Dorna

robot = Dorna()
robot.connect("192.168.254.88")

True

In [8]:
657.36 - 297.36


360.0

In [7]:
robot.iprobe(index=7, val=1)


[3.14209,
 -50.273438,
 122.080078,
 20.192871,
 -37.902832,
 0.219727,
 -14.79375,
 657.36]

# init

In [1]:
# scripts/run_workspace.py
from workspace import Workspace
from workspace.recipes.tool_changer import ToolChanger
from workspace.recipes.hotel import Hotel
from workspace.recipes.adapter import Adapter
from workspace.recipes.feeder import Feeder
from workspace.recipes.rack import Rack
from workspace.recipes.decapper import Decapper


# workspace
workspace = Workspace(config_path="config.j2")
core = workspace.components["core"]

🟢 simulation api enabled
[Display] socket.io connected
[Display] sending initial snapshot (418 items)
[Display] Running at 60 fps


# recipes

In [2]:
# tool changer
rcp_plate_gripper = ToolChanger(workspace=workspace, core=core, component=workspace.components["tool_rack_2"], left_approach=True)
rcp_suction_gripper = ToolChanger(workspace=workspace, core=core, component=workspace.components["tool_rack_1"], left_approach=True)
rcp_tube_gripper = ToolChanger(workspace=workspace, core=core, component=workspace.components["tool_rack_0"], left_approach=True)

# hotel
rcp_hotel = Hotel(workspace=workspace, core=core, component=workspace.components["hotel_0"], left_approach=True, base_distance=50)

# rack
rcp_cap_holder = Rack(workspace=workspace, core=core, component=workspace.components["sbs_adapter_1"], left_approach=False, base_distance=50) # 100
rcp_sbs_plate = Rack(workspace=workspace, core=core, component=workspace.components["sbs_adapter_0"], base_distance=50) # 100

# adapter
rcp_sbs_adaptor = Adapter(workspace=workspace, core=core, component=workspace.components["sbs_adapter_0"], left_approach=True) # 300

# feeder
rcp_feeder = Feeder(workspace=workspace, core=core, component=workspace.components["feeder"], left_approach=False) # 100

# decapper
rcp_decapper = Decapper(workspace=workspace, core=core, component=workspace.components["decapper_0"], base_distance=50) # 100

/usr/local/lib/python3.11/dist-packages/dorna2/ik6r_2.py:304: RuntimeWarning: divide by zero encountered in scalar divide
  c3 = (a2*a2*f14 - 2*a2*(-d4*f24 + c4*d6*f24 + c1*(f14**2 + f24**2)) + f14*(-a3**2+d1**2-d4**2-d5**2+2*c4*d4*d6-d6**2+f14**2+f24**2+2*d1*d7*f33+d7**2*f33**2-2*d1*f34 - 2*d7*f33*f34+f34**2) ) / (2*a3*d5*f14)


# main loop

In [3]:
# lelvels
level = 1
# indices
indices = ["A1", "A8", "F1", "F8"]

# pick plate gripper
rcp_plate_gripper.pick()

for i in range(level):
    # pick from level {i} of hotel
    rcp_hotel.pick_from(i)

    # place the sbs plate in
    rcp_sbs_adaptor.place_in()

    # change the gripper to suction
    rcp_plate_gripper.place()
    rcp_suction_gripper.pick()

    # pick from feeder and place in cap holder
    for index in indices:
        # pick from feeder
        rcp_feeder.pick_from(anchor="place")
        # place in cap holder
        rcp_cap_holder.place_in(index)
    
    # change the gripper to tube gripper
    rcp_suction_gripper.place()
    rcp_tube_gripper.pick()

    # capping
    for index in indices:
        # pick tube
        rcp_sbs_plate.pick_from(index)
        # place in decapper
        rcp_decapper.place_in("place")
        # pick cap
        rcp_cap_holder.pick_from(index)
        # capping
        rcp_decapper.cap(exit=False)
        # pick from decapper
        rcp_decapper.pick_from("place", approach=False)
        # back to sbs plate
        rcp_sbs_plate.place_in(index)  
    
    # place the tube gripper
    rcp_tube_gripper.place()
    
    # pick plate gripper
    rcp_plate_gripper.pick()

    # pick from sbs adaptor
    rcp_sbs_adaptor.pick_from(anchor="place")

    # place in level {i} of hotel
    rcp_hotel.place_in(i)

# place plate gripper
rcp_plate_gripper.place()



KeyboardInterrupt: 